# Chirurgie à chaud EN COURS d'un vrai entraînement + analyse causale de la couche greffée

Fusionne trois expériences du plan de session (P1+P2+P3) en une seule, cohérente :

1. **Phase A** : entraîner le char-LM (TinyShakespeare, dim=256/4 têtes/4 couches,
   `batched_attn=true`) pendant 10 000 pas -- la moitié du run de référence de
   `real_llm.ipynb` (déjà validé : val loss finale 1.6229 à 20 000 pas, Δ0.0033
   nats avec le miroir PyTorch).
2. **Baseline** : recherche de circuit d'induction sur texte réel (noms de
   personnages répétés dans TinyShakespeare, ex. "MENENIUS:") sur le modèle à
   4 couches, à mi-parcours -- `greedy_patch_search!`/`backward_prune!` avec le
   nouveau kwarg `metric` (recovery restreinte à une ligne, comme `induction.ipynb`
   mais sur du texte réel plutôt que sur une tâche synthétique).
3. **Chirurgie** : `insert_block!` (avec le nouveau kwarg `batched_attn=true` pour
   rester homogène) insère une 5ème couche EN COURS d'entraînement -- preuve F1
   (texte généré identique bit-à-bit juste avant/après insertion).
4. **Phase B** : poursuite de l'entraînement (5 couches) pour les 10 000 pas
   restants, moments AdamW fusionnés par nom de paramètre (patron F4 de
   `test/test_surgery.jl`), pas de réinitialisation du pas `t` global.
5. **Final** : même recherche de circuit sur les mêmes fenêtres gelées, maintenant
   sur 5 couches -- la nouvelle couche a-t-elle acquis une responsabilité causale
   mesurable ? Verdict honnête contre des critères falsifiables, quel que soit
   le résultat.

Aucune modification de `real_llm.ipynb` (artefact de parité déjà validé, laissé
intact). Deux correctifs `src/` de cette session sont des prérequis directs :
`greedy_patch_search!`/`backward_prune!` ne perdent plus le patch d'un site déjà
retenu quand un candidat amont est testé (bug trouvé et corrigé le 2026-07-10),
et `insert_block!`/les deux fonctions de recherche acceptent maintenant
`batched_attn`/`metric`.

In [1]:
using NeuroDSL, Random, Statistics, Printf, StatsPlots

dev = NeuroDSL.Backend.CUDADevice()
ns = :real_llm_surgery
println("Device: ", dev)

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   cuSPARSE [b26da814-b3bc-49ef-b0ee-c816305aa060]
│   ChainRulesCoreExt [8ccf546c-b5ad-5491-b9b6-b62e2a4db961]
│   GPUArraysSparseArraysExt [555d850b-e4ac-5d9d-bfa0-17bb432b4efb]
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   NVML [611af6d1-644e-4c5d-bd58-854d7d1254b9]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   ZygoteExt [3f48baa7-5843-56cc-8004-b8b725de387b]
│   FluxCUDAExt [dd41ee52-2073-581e-92e8-26baf003f19a]
│   cuSOLVER [887afef0-6a32-4de5-add4-7827692ba8fc]
│   CUDAExt [9e0aca61-9665-56cf-9642-d947ef6fc392]
│   StructArraysAdaptExt [f04e5bcb-ab32-5a64-8b64-c2cc4abec66e]
│   OneHotArrays [0b1bfda6-eb8a-41d2-88d8-f5af5cad476f]
│   CUDACore [bd0ed864-bdfe-4181-a5ed-ce625a5fdea2]
│   ZygoteDistancesExt [5865c103-18d1-586a-9b11-010bbc2260a8]
│   MLUtilsExt [447e3217-c189

Device: NeuroDSL.Backend.CUDADevice()


## 1. Corpus : TinyShakespeare (téléchargé une seule fois, hors ligne ensuite)

In [2]:
using Downloads

const CORPUS_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
const CORPUS_PATH = joinpath(@__DIR__, "data", "tinyshakespeare", "input.txt")

function load_corpus(path::String, url::String)
    if !isfile(path)
        mkpath(dirname(path))
        Downloads.download(url, path)
    end
    text = read(path, String)
    println("Corpus chargé : ", length(text), " caractères depuis ", path)
    return text
end

text = load_corpus(CORPUS_PATH, CORPUS_URL)
println(first(text, 200))

Corpus chargé : 1115394 caractères depuis C:\Users\Nevermind\Desktop\NeuroDSL\notebook\data\tinyshakespeare\input.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 2. Tokenizer caractère (vrai texte -> vrais IDs, pas de BPE, zéro dépendance nouvelle)

In [3]:
function build_char_tokenizer(text::String)
    chars = sort(unique(collect(text)))
    stoi = Dict(c => i for (i, c) in enumerate(chars))
    return chars, stoi
end

encode(text::AbstractString, stoi::Dict{Char,Int}) = [stoi[c] for c in text]
decode(ids::AbstractVector{<:Integer}, chars::Vector{Char}) = String(chars[ids])

chars, stoi = build_char_tokenizer(text)
vocab_size = length(chars)
println("vocab_size = ", vocab_size)
println("10 premiers caractères (triés) = ", chars[1:10])
println("(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)")

vocab_size = 65
10 premiers caractères (triés) = ['\n', ' ', '!', '$', '&', '\'', ',', '-', '.', '3']
(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)


## 3. Split train/validation (90/10 par position -- le val est la FIN du corpus, jamais vue à l'entraînement)

In [4]:
data = encode(text, stoi)
n_total = length(data)
n_train = floor(Int, 0.9 * n_total)
train_ids = data[1:n_train]
val_ids   = data[n_train+1:end]
println("train: ", length(train_ids), " caractères  |  val: ", length(val_ids), " caractères")

train: 1003854 caractères  |  val: 111540 caractères


## 4. Construction du graphe (copie directe de `build_induction_graph`, généralisée à un vrai vocabulaire/contexte)

In [5]:
function build_char_lm_graph(dev, ns::Symbol; vocab_size::Int, dim::Int, n_heads::Int,
                              hidden_dim::Int, n_layers::Int, block_size::Int)
    g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :token_ids, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(vocab_size, dim)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(block_size, dim)(g, :pos_ids, :pos; namespace=ns)
    x = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(x, [tok_emb, pos_emb], :add; namespace=ns))
    # batched_attn=true : les 2 matmuls par tête (Q·Kᵀ, P·V) passent par un seul
    # appel CUBLAS.gemm_strided_batched! au lieu de n_heads appels séparés
    # (src/layers.jl, conçu avec Fable le 2026-07-10) -- qh/kh/vh/sc_h/ao_h
    # restent des nœuds individuellement adressables (vues zero-copy), donc
    # patch_node!/sweep_patch_sites! continuent de fonctionner sans changement.
    # Mesuré +3.9% sur ce pas exact (test/test_batched_attention.jl, 2793/2793 verts).
    out = NeuroDSL.LlamaModel(n_layers, dim, n_heads, hidden_dim; batched_attn=true)(g, x; namespace=ns)
    logits = NeuroDSL.Linear(dim, vocab_size)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return g, logits
end

# ── Hyperparamètres (voir plan : dimensionnés pour rester bien sous le max GPU
# déjà confirmé cette session -- dim=1024 en forward+backward réel) ──────────
block_size = 256
dim        = 256
n_heads    = 4
hidden_dim = 512
n_layers   = 4

g, logits_sym = build_char_lm_graph(dev, ns; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                     hidden_dim=hidden_dim, n_layers=n_layers, block_size=block_size)
ps = NeuroDSL.params(g; namespace=ns)
n_scalars = sum(length(p.value) for p in ps)
println("Graphe : ", length(g.nodes[ns]), " nœuds, ", length(ps), " tenseurs de paramètres, ",
        n_scalars, " scalaires (~", round(n_scalars/1e6, digits=2), "M)")

Graphe : 220 nœuds, 40 tenseurs de paramètres, 2722369 scalaires (~2.72M)


## 5. Échantillonnage de fenêtres réelles + perte de validation + génération autorégressive

In [6]:
function sample_window(rng, ids::Vector{Int}, block_size::Int)
    i = rand(rng, 1:(length(ids) - block_size))
    tokens = ids[i:i+block_size-1]
    labels = ids[i+1:i+block_size]
    return tokens, labels
end

# 64 fenêtres FIXES également espacées dans le split val -- déterministe,
# comparable entre checkpoints. Jamais de backward_graph! ici (pas de fuite
# du val dans les gradients) -- donc `demand_release!` (src/demand_release.jl)
# est sûr : libère les activations intermédiaires au fil du calcul au lieu de
# les garder résidentes jusqu'au prochain train_char_lm! (mêmes résultats,
# vérifié bit-à-bit cette session -- réduit juste le pic VRAM de cet appel).
function val_loss(g::NeuroDSL.NeuroGraph, ns::Symbol; val_ids::Vector{Int}, block_size::Int, n_windows::Int=64)
    max_start = length(val_ids) - block_size
    starts = round.(Int, range(1, max_start, length=n_windows))
    total = 0.0
    for i in starts
        tokens = val_ids[i:i+block_size-1]
        labels = val_ids[i+1:i+block_size]
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand_release!(g, :loss; namespace=ns)
        total += Float64(sum(Array(loss_val)))
    end
    return total / n_windows
end

# Génération autorégressive -- échantillonnage AVEC TEMPÉRATURE (pas argmax) :
# l'argmax sur un char-LM dégénère quasi systématiquement en boucles
# répétitives ("the the the..."), donnant une fausse impression d'échec alors
# que la distribution apprise est bonne. C'est ce que nanoGPT/char-rnn font
# pour leurs démos.
function generate_text(g::NeuroDSL.NeuroGraph, logits_sym::Symbol, ns::Symbol,
                        stoi::Dict{Char,Int}, chars::Vector{Char};
                        seed_text::String="\n", n_chars::Int=300, temperature::Float32=0.8f0,
                        block_size::Int, rng=MersenneTwister(777), use_argmax::Bool=false)
    ctx = encode(seed_text, stoi)
    generated = Char[]
    for _ in 1:n_chars
        window = length(ctx) > block_size ? ctx[end-block_size+1:end] : ctx
        t = length(window)
        NeuroDSL.set!(g, :token_ids, window; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:t); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        row = Array(NeuroDSL.demand_release!(g, logits_sym; namespace=ns))[end, :]
        local next_id
        if use_argmax
            next_id = argmax(row)
        else
            p = exp.((row .- maximum(row)) ./ temperature)
            p ./= sum(p)
            r = rand(rng)
            cum = 0.0f0
            next_id = length(p)
            for (idx, pi) in enumerate(p)
                cum += pi
                if r <= cum
                    next_id = idx
                    break
                end
            end
        end
        push!(ctx, next_id)
        push!(generated, chars[next_id])
    end
    return String(generated)
end

generate_text (generic function with 1 method)

## 6. Vérification de sanité : perte initiale ≈ ln(vocab_size)

In [7]:
rng_check = MersenneTwister(1)
tokens0, labels0 = sample_window(rng_check, train_ids, block_size)
NeuroDSL.set!(g, :token_ids, tokens0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :labels, labels0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
loss0 = Float64(sum(Array(NeuroDSL.demand!(g, :loss; namespace=ns))))
@printf("Perte initiale mesurée : %.4f   (attendu ln(%d) = %.4f)\n", loss0, vocab_size, log(vocab_size))
@assert abs(loss0 - log(vocab_size)) < 0.5 "Perte initiale trop loin de ln(vocab_size) -- vérifier le câblage avant d'entraîner"

println("\n--- Échantillon AVANT tout entraînement (poids aléatoires) ---")
sample_before = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
println(sample_before)

Perte initiale mesurée : 4.3137   (attendu ln(65) = 4.1744)

--- Échantillon AVANT tout entraînement (poids aléatoires) ---
.?tqpi
GWKdi$CVxeOgtCP$M,zpGDJ!tJG$yf&$KrHG$Z&$'
qVgtl:jnpbXCC$tkb-yyJYiPfZxMn?b3prY
cqQUN
yah;a
nqw;MUekumRq
&yy
&oONCGjsJ;rK:XNn&aT-mxjiRXN:bwEWbAPnrcisNsaVgzh
j&X$Ef
!GgXV
;k'MHrt$vgI:nQ,zJVqAVuqsECY:P-uJrsCYJcV
wZx3$IeywCELnUuWq?wv-;voxSm3zQ!uml$-c&zI AMzXDYJGJXCOilS:ojl kRabR;VC!k,kf&pI;JZfoy$v


## 7. Budget de calcul (estimé par Fable, à partir du chronométrage réel de `real_llm.ipynb`)

Référence mesurée : 505.7 s / 20 000 pas = 25.3 ms/pas à 4 couches (GPU RTX A5500).
Estimation pour cette expérience : Phase A (10k pas, 4 couches) ≈ 253 s ; Phase B
(10k pas, 5 couches) ≈ 316 s (borne sup à 31.6 ms/pas) ; les deux recherches de
circuit (gloutonne + élagage, 3 fenêtres, 16 puis 20 candidats) ≈ moins d'une
minute au total (chaque mesure est un `demand!` incrémental + une restauration
par copie de cache, pas un forward complet). **Total estimé : ~12-16 minutes.**
Le coût est dominé par l'entraînement, pas par l'interprétabilité -- c'est un
résultat en soi, cohérent avec la thèse du framework.

## 8. Phase A : 10 000 premiers pas (4 couches)

`train_char_lm!` v2 : accepte maintenant `rng` (objet, pas une graine entière),
`t0` (pas de départ, pour que le compteur AdamW `t` ne se réinitialise jamais à
la frontière de la greffe) et `moments` (`Dict{Symbol,Tuple}` keyé par NOM de
paramètre, patron F4 de `test/test_surgery.jl` -- après `insert_block!`,
`params(g)` peut réordonner, donc réutiliser des `Vector`s positionnels d'une
phase à l'autre serait incorrect). Retourne `moments` et `rng` en plus des
métriques habituelles, pour les transmettre tels quels à la Phase B.

In [8]:
function train_char_lm!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol,
                         stoi::Dict{Char,Int}, chars::Vector{Char};
                         train_ids::Vector{Int}, val_ids::Vector{Int}, block_size::Int,
                         n_steps::Int, lr::Float32=1f-3, b1::Float32=0.9f0, b2::Float32=0.999f0,
                         eps_v::Float32=1f-8, clip::Float32=1f0, wd::Float32=0f0,
                         rng::MersenneTwister=MersenneTwister(123), t0::Int=0,
                         moments::Union{Nothing,Dict{Symbol,Tuple{Any,Any}}}=nothing,
                         val_every::Int=500, sample_steps=())
    dev = g.device
    ps = NeuroDSL.params(g; namespace=ns)
    # Moments keyés par NOM (pas par position) -- patron F4 (test/test_surgery.jl) :
    # un paramètre déjà connu (avant une éventuelle greffe) reprend ses moments
    # exacts ; un paramètre nouveau (apporté par insert_block!) démarre à zéro.
    m1s = Vector{Any}(undef, length(ps))
    m2s = Vector{Any}(undef, length(ps))
    for (i, p) in enumerate(ps)
        if moments !== nothing && haskey(moments, p.name)
            m1s[i], m2s[i] = moments[p.name]
        else
            m1s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
            m2s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
        end
    end
    train_losses = Float64[]
    val_history = Tuple{Int,Float64}[]

    t_start = time()
    for t in (t0+1):(t0+n_steps)
        tokens, labels = sample_window(rng, train_ids, block_size)
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand!(g, :loss; namespace=ns)
        push!(train_losses, Float64(sum(Array(loss_val))))
        NeuroDSL.backward_graph!(g, :loss; namespace=ns)
        NeuroDSL.adamw_step_batched!(dev, [p.value for p in ps], [p.gradient for p in ps],
                                     m1s, m2s, lr, b1, b2, eps_v, t, clip, wd)
        NeuroDSL.invalidate_all!(g; namespace=ns)

        if t % val_every == 0 || t == t0 + 1
            vl = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
            push!(val_history, (t, vl))
            @printf("step %6d | train %.4f | val %.4f | ppl %.2f | bits/char %.3f\n",
                    t, train_losses[end], vl, exp(vl), vl/log(2))
        end
        if t in sample_steps
            s = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
            println("\n--- Échantillon @ pas $t ---\n", s, "\n")
        end
    end
    elapsed = time() - t_start
    final_moments = Dict{Symbol,Tuple{Any,Any}}(p.name => (m1s[i], m2s[i]) for (i, p) in enumerate(ps))

    return (; train_losses, val_history, elapsed, moments=final_moments, rng)
end

rng_A = MersenneTwister(123)
n_steps_A = 10_000
result_A = train_char_lm!(g, ns, logits_sym, stoi, chars;
                           train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                           n_steps=n_steps_A, lr=1f-3, rng=rng_A, t0=0)
@printf("\nPhase A terminée : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_A, result_A.elapsed, 1000*result_A.elapsed/n_steps_A)

step      1 | train 4.2578 | val 3.7679 | ppl 43.29 | bits/char 5.436
step    500 | train 2.5755 | val 2.5599 | ppl 12.93 | bits/char 3.693
step   1000 | train 2.4011 | val 2.5440 | ppl 12.73 | bits/char 3.670
step   1500 | train 2.5098 | val 2.5219 | ppl 12.45 | bits/char 3.638
step   2000 | train 2.4363 | val 2.4942 | ppl 12.11 | bits/char 3.598
step   2500 | train 2.2969 | val 2.3042 | ppl 10.02 | bits/char 3.324
step   3000 | train 1.9706 | val 2.1521 | ppl 8.60 | bits/char 3.105
step   3500 | train 1.9908 | val 2.0212 | ppl 7.55 | bits/char 2.916
step   4000 | train 1.9451 | val 2.0188 | ppl 7.53 | bits/char 2.912
step   4500 | train 1.7785 | val 1.9537 | ppl 7.06 | bits/char 2.819
step   5000 | train 1.7343 | val 1.8587 | ppl 6.42 | bits/char 2.681
step   5500 | train 1.8053 | val 1.8512 | ppl 6.37 | bits/char 2.671
step   6000 | train 1.5459 | val 1.8083 | ppl 6.10 | bits/char 2.609
step   6500 | train 1.5735 | val 1.8467 | ppl 6.34 | bits/char 2.664
step   7000 | train 1.6374 |

## 9. Sélection et gel de 3 fenêtres de test (protocole d'induction sur texte réel)

Vérifié statistiquement (Fable, sur le split val de 111 540 caractères) : 406
des 882 en-têtes de locuteur ont une répétition du même nom dans les 256
caractères suivants (écart médian 122 caractères) -- largement exploitable.
Protocole : fenêtre de `block_size` caractères contenant un nom de personnage
répété deux fois ; corruption d'un caractère `k` positions après le début de
la 1ère occurrence ; cible = la ligne `j` qui, après avoir lu le préfixe
apparié de la 2ème occurrence, prédit le caractère copiable depuis la 1ère.
Les 3 fenêtres au plus grand effet mesuré (`‖Δlogits[j,:]‖` clean vs
corrompu, sur le modèle DÉJÀ entraîné 10 000 pas) sont **gelées** : réutilisées
à l'identique à la baseline et au point final, jamais re-sélectionnées après
la greffe (une re-sélection biaiserait la comparaison).

In [9]:
using LinearAlgebra

# Scanne des fenêtres de block_size caractères du split val, cherche deux
# en-têtes de locuteur (nom en capitales suivi de ":", précédé d'un saut de
# ligne) portant le MÊME nom dans la fenêtre.
function scan_repeated_name_windows(val_ids::Vector{Int}, chars::Vector{Char}, block_size::Int;
                                     n_scan::Int=400, min_name_len::Int=4, k::Int=3)
    candidates = NamedTuple[]
    max_start = length(val_ids) - block_size
    starts = unique(round.(Int, range(1, max_start, length=n_scan)))
    for s in starts
        w = val_ids[s:s+block_size-1]
        wtext = decode(w, chars)
        headers = NamedTuple[]
        for m in eachmatch(r"\n([A-Z][A-Z ]{2,})\:", wtext)
            name = String(strip(m.captures[1]))
            length(name) >= min_name_len || continue
            push!(headers, (; p=m.offset + 1, name))
        end
        length(headers) < 2 && continue
        for i in 1:length(headers)-1, jx in i+1:length(headers)
            headers[i].name != headers[jx].name && continue
            kk = min(k, length(headers[i].name) - 1)
            kk < 1 && continue
            p1, p2 = headers[i].p, headers[jx].p
            p2 + kk > block_size && continue
            push!(candidates, (; window_start=s, p1, p2, k=kk, name=headers[i].name))
        end
    end
    return candidates
end

candidates_raw = scan_repeated_name_windows(val_ids, chars, block_size)
println("Fenêtres candidates avec nom de personnage répété : ", length(candidates_raw))
@assert length(candidates_raw) >= 3 "Pas assez de fenêtres candidates -- revoir le protocole avant de continuer"

# Taille d'effet (‖Δlogits[j,:]‖ clean vs corrompu) mesurée sur le modèle DÉJÀ
# entraîné (Phase A) -- un effet non nul prouve que le modèle a appris, À CE
# STADE, une dépendance comportementale réelle à cette position.
function effect_size(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j)
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupt_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    return norm(clean_logits[j, :] .- corrupt_logits[j, :])
end

scored = NamedTuple[]
rng_scan = MersenneTwister(999)
for c in candidates_raw
    tokens_clean = val_ids[c.window_start:c.window_start+block_size-1]
    j = c.p2 + c.k - 1
    (j < 1 || j > block_size) && continue
    tokens_corrupt = copy(tokens_clean)
    orig_id = tokens_corrupt[c.p1 + c.k]
    new_id = orig_id
    while new_id == orig_id
        new_id = rand(rng_scan, 1:vocab_size)
    end
    tokens_corrupt[c.p1 + c.k] = new_id
    eff = effect_size(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j)
    push!(scored, (; c..., j, tokens_clean, tokens_corrupt, effect=eff))
end

sort!(scored, by = x -> -x.effect)
frozen_windows = scored[1:min(3, length(scored))]
println("\nFenêtres GELÉES (top taille d'effet, jamais re-sélectionnées après la greffe) :")
for fw in frozen_windows
    @printf("  nom=%-14s window_start=%-7d p1=%-4d p2=%-4d k=%d  j=%-4d  effet=%.4f\n",
            fw.name, fw.window_start, fw.p1, fw.p2, fw.k, fw.j, fw.effect)
end

Fenêtres candidates avec nom de personnage répété : 196

Fenêtres GELÉES (top taille d'effet, jamais re-sélectionnées après la greffe) :
  nom=PETRUCHIO      window_start=10320   p1=70   p2=129  k=3  j=131   effet=4.8296
  nom=PETRUCHIO      window_start=23987   p1=59   p2=150  k=3  j=152   effet=3.4510
  nom=PETRUCHIO      window_start=47973   p1=120  p2=188  k=3  j=190   effet=3.1458


## 10. Recherche de circuit — BASELINE (avant greffe, 4 couches, pas 10 000)

`greedy_patch_search!` + `backward_prune!` (patch de tête ENTIÈRE, comme
`induction.ipynb` -- une tête est un canal étroit, pas de récupération
triviale à 100%), avec le nouveau kwarg `metric` restreignant la mesure à la
ligne cible `j` de chaque fenêtre gelée. Sert de point de comparaison "avant
greffe" pour la Section 13.

In [10]:
"""
    find_circuit!(g, ns, logits_sym, window; max_sites=6)

Recherche de circuit sur une fenêtre gelée : capture les caches propre/corrompu
LOCALEMENT (variables locales, jamais de globale réutilisée entre appels --
aucune fuite possible d'un cache périmé après une mutation de graphe entre deux
appels), candidats = toutes les sorties de tête `*_mha_ao_h{h}` du graphe
COURANT (16 avant greffe, 20 après), métrique = recovery restreinte à la ligne
`window.j` (via le nouveau kwarg `metric`, sinon la sortie entière noierait un
effet localisé à une seule position dans une séquence de 256 caractères).
`greedy_patch_search!`/`backward_prune!` réappliquent maintenant `selected` à
chaque mutation (correctif du 2026-07-10) -- sûr même si un site tardif est
retenu avant qu'un site précoce ne soit testé, exactement le cas d'un vrai
circuit d'induction. Remet le graphe en état propre avant de retourner.
"""
function find_circuit!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol, window; max_sites::Int=6)
    tokens_clean, tokens_corrupt, j = window.tokens_clean, window.tokens_corrupt, window.j

    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    clean_cache  = NeuroDSL.capture_activations(g, ns)   # demand! (pas demand_release!) -- capture_activations a besoin des valeurs intermédiaires

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupted_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    corrupted_cache  = NeuroDSL.capture_activations(g, ns)

    row_metric(out) = NeuroDSL.recovery_metric(Array(out)[j:j, :], clean_output[j:j, :], corrupted_output[j:j, :])

    candidates = sort(collect(filter(s -> occursin(r"_mha_ao_h\d+$", String(s)), keys(g.nodes[ns]))))

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    selected, trajectory = NeuroDSL.greedy_patch_search!(g, logits_sym, candidates, clean_cache, corrupted_cache,
                                                           clean_output, corrupted_output;
                                                           namespace=ns, max_sites=max_sites, metric=row_metric)
    remaining, pruned = if isempty(selected)
        (Symbol[], Symbol[])
    else
        NeuroDSL.backward_prune!(g, logits_sym, selected, clean_cache, corrupted_cache,
                                  clean_output, corrupted_output; namespace=ns, metric=row_metric)
    end

    # Vérification indépendante (ancre de sanité) : recovery du sous-ensemble
    # final recalculée depuis un état frais, via patch_nodes! direct.
    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    isempty(remaining) || NeuroDSL.patch_nodes!(g, remaining, clean_cache; namespace=ns)
    out_check = NeuroDSL.demand!(g, logits_sym; namespace=ns)
    r_check = row_metric(out_check)

    # Remise en état propre (texte clean) pour la suite du notebook.
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    return (; candidates, selected, trajectory, remaining, pruned, r_check,
            n_candidates=length(candidates), clean_cache, corrupted_cache, clean_output, corrupted_output)
end

baseline_results = [find_circuit!(g, ns, logits_sym, w) for w in frozen_windows]

println("Recherche de circuit -- BASELINE (4 couches, pas ", n_steps_A, ") :")
for (w, r) in zip(frozen_windows, baseline_results)
    println("  fenêtre '", w.name, "' : sites retenus après élagage = ", r.remaining,
            "  recovery vérifiée = ", round(r.r_check, digits=4), "  (", r.n_candidates, " candidats)")
end

Recherche de circuit -- BASELINE (4 couches, pas 10000) :
  fenêtre 'PETRUCHIO' : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_4_mha_ao_h2, :layer_1_mha_ao_h1, :layer_1_mha_ao_h3]  recovery vérifiée = 0.9497  (16 candidats)
  fenêtre 'PETRUCHIO' : sites retenus après élagage = [:layer_1_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_1_mha_ao_h4, :layer_2_mha_ao_h4, :layer_2_mha_ao_h3]  recovery vérifiée = 0.9996  (16 candidats)
  fenêtre 'PETRUCHIO' : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_1_mha_ao_h4, :layer_1_mha_ao_h3, :layer_3_mha_ao_h1]  recovery vérifiée = 0.9986  (16 candidats)


## 11. Chirurgie à chaud — insertion d'une 5ème couche EN COURS d'entraînement

`insert_block!` insère un nouveau `LlamaBlock` (`batched_attn=true`, homogène
avec le reste du modèle) juste après `:layer_2_out`, initialisé pour calculer
l'IDENTITÉ EXACTE (poids de sortie attention + dernière matrice MLP mis à
zéro -- voir la docstring de `src/graph_surgery.jl`). Preuve F1 : texte généré
ET val loss comparés bit-à-bit/exactement juste avant et juste après
l'insertion -- doivent être strictement identiques, pas seulement proches.

In [11]:
n_params_before_graft = length(NeuroDSL.params(g; namespace=ns))

println("--- Échantillon JUSTE AVANT insertion (4 couches) ---")
sample_before_graft = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                                     rng=MersenneTwister(777), n_chars=200)
val_before_graft = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
println(sample_before_graft)
@printf("val loss juste avant insertion : %.8f\n", val_before_graft)

# ── La chirurgie elle-même : une ligne. ──────────────────────────────────
new_out = NeuroDSL.insert_block!(g, ns, :layer_2_out, dim, n_heads, hidden_dim; batched_attn=true)
println("\nCouche insérée après :layer_2_out -> nouveau symbole de sortie : ", new_out)

println("\n--- Échantillon JUSTE APRÈS insertion (5 couches) ---")
sample_after_graft = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                                    rng=MersenneTwister(777), n_chars=200)
val_after_graft = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
println(sample_after_graft)
@printf("val loss juste après insertion  : %.8f\n", val_after_graft)

# ── Preuve F1 : identité exacte, pas approximative. ──────────────────────
@assert sample_before_graft == sample_after_graft "F1 violée : le texte généré a changé après une insertion censée être l'identité exacte"
@assert val_before_graft == val_after_graft "F1 violée : la val loss a changé après l'insertion"
println("\n✅ F1 confirmée : texte généré ET val loss strictement identiques avant/après insertion.")

n_params_after_graft = length(NeuroDSL.params(g; namespace=ns))
println("Paramètres : ", n_params_before_graft, " -> ", n_params_after_graft,
        "  (+", n_params_after_graft - n_params_before_graft, " nouveaux, apportés par la greffe)")

--- Échantillon JUSTE AVANT insertion (4 couches) ---
Althis heariad with go have end;
Betten undaway 'twere to-morrow.

Sicitizens:
O the prisonous o' the nother ' to be in the
that the botter of when livest the woat o'
s all o cold to the that not to f
val loss juste avant insertion : 1.71786640

Couche insérée après :layer_2_out -> nouveau symbole de sortie : surgery_layer_2_out_out

--- Échantillon JUSTE APRÈS insertion (5 couches) ---
Althis heariad with go have end;
Betten undaway 'twere to-morrow.

Sicitizens:
O the prisonous o' the nother ' to be in the
that the botter of when livest the woat o'
s all o cold to the that not to f
val loss juste après insertion  : 1.71786640

✅ F1 confirmée : texte généré ET val loss strictement identiques avant/après insertion.
Paramètres : 40 -> 49  (+9 nouveaux, apportés par la greffe)


## 12. Phase B — 10 000 pas restants (5 couches)

`rng`/`moments` transmis tels quels depuis la Phase A (`result_A.rng`,
`result_A.moments`), `t0=n_steps_A` -- le compteur AdamW global continue sans
jamais repasser à 1 (pas de faux "warmup" pour les 4 couches d'origine à la
frontière). Les nouveaux paramètres de la couche greffée démarrent avec des
moments à zéro, exactement le patron F4 déjà validé (`test/test_surgery.jl` :
le modèle greffé réapprend réellement après la greffe).

In [12]:
n_steps_B = 10_000
result_B = train_char_lm!(g, ns, logits_sym, stoi, chars;
                           train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                           n_steps=n_steps_B, lr=1f-3, rng=result_A.rng, t0=n_steps_A,
                           moments=result_A.moments)
@printf("\nPhase B terminée : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_B, result_B.elapsed, 1000*result_B.elapsed/n_steps_B)

final_val_loss = result_B.val_history[end][2]
@printf("Val loss finale (pas %d, 5 couches) : %.4f nats/char  (ppl %.2f)\n",
        n_steps_A + n_steps_B, final_val_loss, exp(final_val_loss))
println("(Référence 4 couches/20000 pas, real_llm.ipynb : val loss finale 1.6229)")

step  10001 | train 1.4428 | val 1.8182 | ppl 6.16 | bits/char 2.623
step  10500 | train 1.5629 | val 1.7316 | ppl 5.65 | bits/char 2.498
step  11000 | train 1.4755 | val 1.7355 | ppl 5.67 | bits/char 2.504
step  11500 | train 1.5979 | val 1.7131 | ppl 5.55 | bits/char 2.472
step  12000 | train 1.5375 | val 1.6975 | ppl 5.46 | bits/char 2.449
step  12500 | train 1.4438 | val 1.7245 | ppl 5.61 | bits/char 2.488
step  13000 | train 1.6524 | val 1.6964 | ppl 5.45 | bits/char 2.447
step  13500 | train 1.4780 | val 1.6831 | ppl 5.38 | bits/char 2.428
step  14000 | train 1.2285 | val 1.6923 | ppl 5.43 | bits/char 2.441
step  14500 | train 1.5754 | val 1.6688 | ppl 5.31 | bits/char 2.408
step  15000 | train 1.4093 | val 1.6897 | ppl 5.42 | bits/char 2.438
step  15500 | train 1.4503 | val 1.6851 | ppl 5.39 | bits/char 2.431
step  16000 | train 1.4806 | val 1.6768 | ppl 5.35 | bits/char 2.419
step  16500 | train 1.3752 | val 1.6768 | ppl 5.35 | bits/char 2.419
step  17000 | train 1.6269 | val 1

## 13. Recherche de circuit — FINAL (5 couches) + comparaison

Mêmes 3 fenêtres GELÉES (jamais re-sélectionnées après la greffe -- une
re-sélection biaiserait la comparaison), mêmes corruptions, `find_circuit!`
appelé sur le graphe maintenant à 5 couches (20 têtes candidates au lieu de
16). Les caches propre/corrompu sont recapturés localement à cet instant
précis (portée lexicale de la fonction) -- aucune fuite possible depuis les
caches de la baseline, qui référençaient une topologie de graphe désormais
périmée.

In [13]:
final_results = [find_circuit!(g, ns, logits_sym, w) for w in frozen_windows]

println("\n" * "="^70)
println("COMPARAISON BASELINE (4 couches, pas ", n_steps_A, ") vs FINAL (5 couches, pas ", n_steps_A+n_steps_B, ")")
println("="^70)
for (w, rb, rf) in zip(frozen_windows, baseline_results, final_results)
    println("\nFenêtre '", w.name, "' (taille d'effet à la baseline = ", round(w.effect, digits=4), ")")
    println("  baseline : sites retenus après élagage = ", rb.remaining,
            "  recovery=", round(rb.r_check, digits=4), "  (", rb.n_candidates, " candidats testés)")
    println("  final    : sites retenus après élagage = ", rf.remaining,
            "  recovery=", round(rf.r_check, digits=4), "  (", rf.n_candidates, " candidats testés)")
    grafted_in_final = filter(s -> occursin("surgery_", String(s)), rf.remaining)
    println("  têtes greffées retenues : ", isempty(grafted_in_final) ? "aucune" : grafted_in_final)
end


COMPARAISON BASELINE (4 couches, pas 10000) vs FINAL (5 couches, pas 20000)

Fenêtre 'PETRUCHIO' (taille d'effet à la baseline = 4.8296)
  baseline : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_4_mha_ao_h2, :layer_1_mha_ao_h1, :layer_1_mha_ao_h3]  recovery=0.9497  (16 candidats testés)
  final    : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_1_mha_ao_h3, :layer_1_mha_ao_h4, :layer_2_mha_ao_h2]  recovery=0.9958  (20 candidats testés)
  têtes greffées retenues : aucune

Fenêtre 'PETRUCHIO' (taille d'effet à la baseline = 3.451)
  baseline : sites retenus après élagage = [:layer_1_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_1_mha_ao_h4, :layer_2_mha_ao_h4, :layer_2_mha_ao_h3]  recovery=0.9996  (16 candidats testés)
  final    : sites retenus après élagage = [:layer_1_mha_ao_h1, :layer_1_mha_ao_h3, :layer_1_mha_ao_h2, :layer_1_mha_ao_h4, :layer_2_mha_ao_h2, :layer_3_mha_ao_h3]  recovery=0.9978  (20 candidats

## 14. Verdict : la couche greffée a-t-elle acquis un rôle causal ?

Critères falsifiables (Fable) : **positif** = normes de sortie greffées non
nulles ET au moins une tête greffée retenue après élagage sur ≥2/3 fenêtres
avec contribution marginale ≥0.05 ET signature d'attention interprétable ;
**négatif** = normes ~0 ou aucune tête greffée jamais retenue, trajectoires
≈ baseline ; **mitigé** = retenue puis élaguée, ou contribution sous le
seuil. Rapporté tel quel, quel que soit le résultat.

In [14]:
graft_prefix = :surgery_layer_2_out
output_W_sym = Symbol(graft_prefix, :_mha_output_W)
mlp_w2_sym   = Symbol(graft_prefix, :_mlp_w2)
norm_output_W = norm(Array(NeuroDSL.node(g, output_W_sym; namespace=ns).value))
norm_mlp_w2   = norm(Array(NeuroDSL.node(g, mlp_w2_sym; namespace=ns).value))
orig_norms_output_W = [norm(Array(NeuroDSL.node(g, Symbol(:layer_,i,:_mha_output_W); namespace=ns).value)) for i in 1:4]

println("‖output_W‖ greffé = ", round(norm_output_W, digits=4),
        "   (moyenne des 4 couches d'origine = ", round(mean(orig_norms_output_W), digits=4), ")")
println("‖mlp_w2‖ greffé   = ", round(norm_mlp_w2, digits=4))

# Sweep individuel des 4 têtes greffées sur la dernière fenêtre gelée -- combien
# chaque tête greffée récupère-t-elle SEULE, indépendamment de l'élagage collectif ?
w_last  = frozen_windows[end]
rf_last = final_results[end]
row_metric_last(out) = NeuroDSL.recovery_metric(Array(out)[w_last.j:w_last.j, :],
                                                 rf_last.clean_output[w_last.j:w_last.j, :],
                                                 rf_last.corrupted_output[w_last.j:w_last.j, :])
grafted_heads = filter(s -> occursin(r"surgery_layer_2_out_mha_ao_h\d+$", String(s)), rf_last.candidates)

println("\nSweep individuel des têtes greffées (fenêtre '", w_last.name, "') :")
NeuroDSL.set!(g, :token_ids, w_last.tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
NeuroDSL.demand!(g, logits_sym; namespace=ns)
head_recoveries = Dict{Symbol,Float64}()
for h in grafted_heads
    NeuroDSL.patch_node!(g, h, rf_last.clean_cache; namespace=ns)
    out = NeuroDSL.demand!(g, logits_sym; namespace=ns)
    r = row_metric_last(out)
    head_recoveries[h] = r
    println("  ", h, " seule : recovery = ", round(r, digits=4))
    NeuroDSL.patch_node!(g, h, rf_last.corrupted_cache; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
end
NeuroDSL.set!(g, :token_ids, w_last.tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
NeuroDSL.demand!(g, logits_sym; namespace=ns)

# ── Verdict, contre les critères falsifiables de la cellule précédente ──────
any_grafted_retained = any(!isempty(filter(s -> occursin("surgery_", String(s)), rf.remaining)) for rf in final_results)
weights_left_zero = norm_output_W < 1f-3 && norm_mlp_w2 < 1f-3
best_head_recovery = isempty(head_recoveries) ? 0.0 : maximum(values(head_recoveries))

println("\n" * "="^70)
println("VERDICT")
println("="^70)
println("Poids de sortie greffés restés à zéro (jamais entraînés) ? ", weights_left_zero)
println("Au moins une tête greffée retenue après élagage (une fenêtre au moins) ? ", any_grafted_retained)
println("Meilleure recovery individuelle d'une tête greffée (dernière fenêtre) : ", round(best_head_recovery, digits=4))

verdict = if weights_left_zero
    "NÉGATIF -- la couche greffée n'a jamais quitté l'identité (poids de sortie encore à zéro) : dormante pour cette capacité."
elseif any_grafted_retained && best_head_recovery >= 0.05
    "POSITIF -- au moins une tête greffée est retenue après élagage avec une contribution mesurable."
elseif any_grafted_retained
    "MITIGÉ -- une tête greffée a été sélectionnée par la recherche gloutonne mais élaguée, ou sa contribution individuelle reste sous le seuil de 0.05."
else
    "NÉGATIF -- la couche a quitté l'identité (poids non nuls) mais n'a acquis aucun rôle causal détectable dans ce circuit précis."
end
println("\n", verdict)

‖output_W‖ greffé = 13.927   (moyenne des 4 couches d'origine = 22.4297)
‖mlp_w2‖ greffé   = 12.8623

Sweep individuel des têtes greffées (fenêtre 'PETRUCHIO') :
  surgery_layer_2_out_mha_ao_h1 seule : recovery = -0.0009
  surgery_layer_2_out_mha_ao_h2 seule : recovery = 0.0008
  surgery_layer_2_out_mha_ao_h3 seule : recovery = -0.0029
  surgery_layer_2_out_mha_ao_h4 seule : recovery = -0.0011

VERDICT
Poids de sortie greffés restés à zéro (jamais entraînés) ? false
Au moins une tête greffée retenue après élagage (une fenêtre au moins) ? false
Meilleure recovery individuelle d'une tête greffée (dernière fenêtre) : 0.0008

NÉGATIF -- la couche a quitté l'identité (poids non nuls) mais n'a acquis aucun rôle causal détectable dans ce circuit précis.


## 15. Courbes combinées (Phase A + Phase B, frontière de la greffe marquée)

In [15]:
train_losses_all = vcat(result_A.train_losses, result_B.train_losses)
val_history_all  = vcat(result_A.val_history, result_B.val_history)

plot(1:length(train_losses_all), train_losses_all, label="train loss (par pas)",
     alpha=0.4, color=:steelblue, xlabel="pas", ylabel="perte (nats/char)",
     title="NeuroDSL -- char-LM sur TinyShakespeare, chirurgie à chaud @ pas $(n_steps_A)")
val_x = [v[1] for v in val_history_all]
val_y = [v[2] for v in val_history_all]
plot!(val_x, val_y, label="val loss", color=:orange, lw=2, marker=:circle, markersize=3)
vline!([n_steps_A], label="insertion de la couche", linestyle=:dot, color=:red, lw=2)
hline!([log(vocab_size)], label="ln(vocab_size) -- niveau aléatoire", linestyle=:dash, color=:gray)
savefig(joinpath(@__DIR__, "..", "figures", "real_llm_surgery_loss.pdf"))
println("Figure sauvegardée -> figures/real_llm_surgery_loss.pdf")

Figure sauvegardée -> figures/real_llm_surgery_loss.pdf


## 16. Sauvegarde des résultats

Toutes les métriques clés de l'expérience -- Phase A/B, preuve F1, baseline vs
final, verdict -- dans un fichier JSON séparé (`real_llm_surgery.ipynb` ne
touche jamais `real_llm_neurodsl_results.json`, l'artefact de `real_llm.ipynb`).

In [16]:
using JSON

surgery_results = Dict(
    "vocab_size" => vocab_size,
    "n_steps_A" => n_steps_A,
    "n_steps_B" => n_steps_B,
    "elapsed_A_s" => result_A.elapsed,
    "elapsed_B_s" => result_B.elapsed,
    "val_loss_mid_training" => result_A.val_history[end][2],
    "val_loss_final" => final_val_loss,
    "val_loss_reference_4layers_20k" => 1.6229,   # real_llm.ipynb, session précédente
    "f1_identity_confirmed" => (sample_before_graft == sample_after_graft) && (val_before_graft == val_after_graft),
    "n_params_before_graft" => n_params_before_graft,
    "n_params_after_graft" => n_params_after_graft,
    "frozen_windows" => [(; name=w.name, effect=w.effect, j=w.j) for w in frozen_windows],
    "baseline_selected" => [String.(r.remaining) for r in baseline_results],
    "baseline_recovery" => [r.r_check for r in baseline_results],
    "final_selected" => [String.(r.remaining) for r in final_results],
    "final_recovery" => [r.r_check for r in final_results],
    "grafted_heads_retained_any_window" => any_grafted_retained,
    "norm_output_W_grafted" => norm_output_W,
    "norm_mlp_w2_grafted" => norm_mlp_w2,
    "norm_output_W_original_mean" => mean(orig_norms_output_W),
    "grafted_head_sweep" => Dict(String(k) => v for (k,v) in head_recoveries),
    "verdict" => verdict,
)
open(joinpath(@__DIR__, "real_llm_surgery_results.json"), "w") do io
    JSON.print(io, surgery_results)
end
println("Résultats écrits -> notebook/real_llm_surgery_results.json")

Résultats écrits -> notebook/real_llm_surgery_results.json
